In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client()
dataset_ref = client.dataset("epa_historical_air_quality", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = list(client.list_tables(dataset))

print("Tables in the EPA dataset:")
for table in tables:
    print(table.table_id)

In [ ]:
table_ref = dataset_ref.table("air_quality_annual_summary")
table = client.get_table(table_ref)

for schema_field in table.schema:
    print(f"{schema_field.name} - {schema_field.field_type}")

In [ ]:
# Extracting the arithmetic mean of particulates below 2.5
# these particulates stem from vehicle and building emissions
# that can be generated by burning, chemical reactions, dust storms
# they can also be secondarily generated in the atmosphere through chemical reactions

query = """
SELECT 
    date_local, 
    arithmetic_mean as pm25
FROM 
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_daily_summary`
WHERE 
    state_name = 'California' 
    AND county_name = 'Los Angeles'
    AND date_local >= '2018-01-01' AND date_local <= '2025-12-31'
ORDER BY 
    date_local ASC
"""

query_job = client.query(query)
 
df = query_job.to_dataframe()

display(df.head())

I'm changing the format to pandas datetime just to make sure it is handled properly

I will also be printing the dataframe info to make sure I do not have to deal with empty or null values since it is a time series

In [ ]:
# reformatting the data and averaging county data measurements
df['date_local'] = pd.to_datetime(df['date_local'])

# date is now the index so we can query using a specified datetime
df.set_index('date_local', inplace=True)

print(df.info())

## Preprocessing
Since the data is a time series, we need to ensure that it is:
- Contiguous
- Sorted
- Free of missing values (done)

I will proceed to visualize the dataset to look for 
- Seasonality (repeating patterns)
- Trend (arrangement of datpoints following a pattern)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.plot(df.index, df['pm25'], label='Daily PM2.5', linewidth=1)
plt.title('Daily Average PM2.5 in LA (2018-2025)')
plt.xlabel('Date')
plt.ylabel('PM2.5 Concentration (µg/m³)')
plt.legend()
plt.show()

That is how the overall data looks but there are two problems: 
- sample is too large, so I will reduce it to a random year to see if there might be seasonal changes (like summer wildfires)
- Datapoints from 2022 - 2023, 2023 - 2024 and 2024-2025 look suspiciously wrong 

First I will plot two years that don't look suspicious to see if there is seasonality or trend within them. I'll define a function to plot a range for this PDA.

In [ ]:
def plot_date_range(df, start_date, end_date, column='pm25'):
    date_range_subset = df.loc[pd.to_datetime(start_date):pd.to_datetime(end_date)]
    
    plt.figure(figsize=(15, 6))
    plt.plot(date_range_subset.index, date_range_subset[column], linewidth=1.5)
    
    plt.title(f'Daily Average PM2.5 in Los Angeles ({start_date} to {end_date})')
    plt.xlabel('Date')
    plt.ylabel('PM2.5 Concentration (µg/m³)')
    plt.tight_layout()
    plt.show()

plot_date_range(df, '2018-01-01','2018-12-31')

In [ ]:
plot_date_range(df, '2019-01-01','2019-12-31')

In [ ]:
plot_date_range(df, '2020-01-01','2020-12-31')

There seems to be some seasonality on the last months and the beginning months (probably due to increased electric consumption in winter) and an insane peak in July (fireworks due to July 4th)

I will now make boxplots for the seasons to make the visualization clearer

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

def plot_seasonal_buckets(df, start_date, end_date, column='pm25'):
    subset = df.loc[start_date:end_date].copy()
    
    subset['season'] = subset.index.month.map(get_season)
    plt.figure(figsize=(10, 6))
    season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
    
    sns.boxplot(data=subset, x='season', y=column, order=season_order)
    
    plt.title(f'Seasonal PM2.5 Buckets in Los Angeles ({start_date} to {end_date})')
    plt.xlabel('Season')
    plt.ylabel('PM2.5 Concentration (µg/m³)')
    plt.tight_layout()
    plt.show()

plot_seasonal_buckets(df, '2018-01-01', '2018-12-31')

In [ ]:
# the following code was generated by Gemini 3.1 pro
# why? i needed a complex graph and I am not used to the seaborn package
# it is also just for EDA

# 1. Create a copy and extract the necessary categorical columns
df_subplots = df.copy()
df_subplots['year'] = df_subplots.index.year
df_subplots['season'] = df_subplots.index.month.map(get_season)

# 2. Define the logical order for the x-axis
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']

# 3. Use catplot to generate the subplots
# 'col="year"' tells Seaborn to make a new subplot for each unique year
# 'col_wrap=3' limits the grid to 3 subplots per row before wrapping to the next line
g = sns.catplot(
    data=df_subplots, 
    x='season', 
    y='pm25', 
    col='year', 
    col_wrap=3, 
    kind='box', 
    order=season_order,
    height=4, 
    aspect=1.2,
    palette='Set2'
)

# 4. Format the titles and labels
g.set_axis_labels("Season", "PM2.5 Concentration (µg/m³)")
g.set_titles("Year: {col_name}")

# Add a main title to the entire figure (y=1.05 pushes it slightly above the subplots)
g.fig.suptitle("Seasonal Distribution of PM2.5 in Los Angeles (2018-2022)", y=1.05)

plt.show()

Years 2022 and onwards look incomplete, so I will only use the data from 2018-2021.

Also there is a clear indication that Winter has the largest variation and ranges for PM 2.5, so this set does possess seasonality (as well as in Autumn)!

## Data split
Since the data from 2022 seems to be incomplete, I will split the data using the classic 80 - 20 split from 2018 to 2021

In [ ]:
filtered_data = df.loc['2018-01-01':'2021-12-31']

training_fraction = 0.8

# index where the split happens
# int casting because it's supposed to be an idx
split_index = int(training_fraction * len(filtered_data))

train_data = filtered_data.iloc[:split_index]
test_data = filtered_data.iloc[split_index:]

Finally we rescale the data to go from 0.0 to 1.0 for training

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_data[['pm25']])
test_scaled = scaler.transform(test_data[['pm25']])